# LSTM 90-day daily feature generation

Creates one 90-day sequence for every `(login_id, acct_nbr, anchor_date)` sample.

Outputs:
- `daily_features_long`: 90 rows per sample, 20 features per row.
- `lstm_sequences`: one row per sample with a `90 x 20` nested array.

`balance_std_7d` is intentionally excluded. The two touch/sequence ratios use different denominators: one compares employees on an account; the other compares accounts for an employee.

In [ ]:
%run ../utils/load_tables_utils

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

WINDOW_DAYS = 90
BALANCE_STD_DAYS = 30
WRITE_OUTPUT = False

input_artifact_root = (
    'abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/'
    'ins_us_nms/v1'
)
output_root = (
    'abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/'
    'mrm/ins_us_nms'
)
windows_path = f'{input_artifact_root}/output/insider_us_nms_windows_v2'
long_output_path = f'{output_root}/output/insider_us_nms_lstm_daily_features_v1'
sequence_output_path = f'{output_root}/output/insider_us_nms_lstm_sequences_v1'

model_windows = spark.read.format('delta').load(windows_path)

FEATURE_COLS = [
    'daily_inquiries',
    'daily_maintenances',
    'daily_prop_inquiries',
    'daily_prop_maintenances',
    'daily_after_hours_touches',
    'daily_emp_prop_acct_seqs',
    'daily_avg_gap_minutes',
    'daily_acct_balance',
    'daily_balance_change',
    'daily_status_change',
    'daily_sequence_duration_seconds',
    'balance_std_30d',
    'employee_daily_avg_touches_per_account',
    'daily_acct_balance_touch_deviation',
    'daily_emp_touch_seq_ratio',
    'daily_stddev_acct_balance_emp',
    'daily_acct_touch_seq_ratio',
    'daily_contact_info_changes',
    'daily_emp_behavior_deviation',
    'daily_address_changes',
]

assert len(FEATURE_COLS) == 20

def safe_div(numerator, denominator):
    return (
        F.when(denominator.isNull() | (denominator == 0), F.lit(0.0))
        .otherwise(numerator.cast('double') / denominator.cast('double'))
    )

## 1. Samples and complete 90-day calendar

The existing window `date` is renamed to `anchor_date`. Each sample contains calendar dates from `anchor_date - 89` through `anchor_date`, inclusive.

In [ ]:
metadata_candidates = [
    'lookback_window_start', 'fraud_date', 'label', 'insider_label',
    'label_split', 'cv_fold', 'job_family_description', 'party_ids'
]
metadata_cols = [c for c in metadata_candidates if c in model_windows.columns]

samples = (
    model_windows
    .select(
        F.upper('login_id').alias('login_id'),
        F.col('acct_nbr').cast('string').alias('acct_nbr'),
        F.to_date('date').alias('anchor_date'),
        *metadata_cols,
    )
    .dropDuplicates(['login_id', 'acct_nbr', 'anchor_date'])
)

sample_key = ['login_id', 'acct_nbr', 'anchor_date']

spine = (
    samples
    .withColumn(
        'calendar_date',
        F.explode(
            F.sequence(
                F.date_sub('anchor_date', WINDOW_DAYS - 1),
                F.col('anchor_date'),
            )
        ),
    )
    .withColumn('sequence_day', F.datediff('calendar_date', F.date_sub('anchor_date', WINDOW_DAYS - 1)))
)

spine.select(sample_key + ['calendar_date', 'sequence_day']).display()

## 2. Employee-account daily transaction features

A sequence is one employee-account-day containing at least one touch. Inquiry and maintenance counts use transaction types 3 and 4; proportions divide those counts by all touches on that day.

In [ ]:
trx = (
    df_trx
    .select(
        F.upper('login_id').alias('login_id'),
        F.col('acct_nbr').cast('string').alias('acct_nbr'),
        F.to_date('transaction_date').alias('calendar_date'),
        F.col('transaction_datetime').cast('timestamp').alias('transaction_datetime'),
        F.col('transaction_type').cast('string').alias('transaction_type'),
    )
    .dropna(subset=['login_id', 'acct_nbr', 'calendar_date', 'transaction_datetime'])
)

# Spark dayofweek: Sunday=1, ..., Saturday=7. This is the original schedule.
business_hours = spark.createDataFrame(
    [(1, 9, 30, 15, 30), (2, 7, 0, 18, 30), (3, 7, 0, 18, 30),
     (4, 7, 0, 18, 30), (5, 7, 0, 19, 30), (6, 7, 0, 19, 30),
     (7, 7, 30, 14, 30)],
    ['day_of_week', 'start_hour', 'start_minute', 'end_hour', 'end_minute'],
)

trx_enriched = (
    trx
    .withColumn('day_of_week', F.dayofweek('calendar_date'))
    .join(F.broadcast(business_hours), 'day_of_week')
    .withColumn('trx_minute', F.hour('transaction_datetime') * 60 + F.minute('transaction_datetime'))
    .withColumn('start_time', F.col('start_hour') * 60 + F.col('start_minute'))
    .withColumn('end_time', F.col('end_hour') * 60 + F.col('end_minute'))
    .withColumn('is_after_hours', ((F.col('trx_minute') < F.col('start_time')) | (F.col('trx_minute') > F.col('end_time'))).cast('int'))
)

gap_window = Window.partitionBy('login_id', 'acct_nbr', 'calendar_date').orderBy('transaction_datetime')
trx_enriched = (
    trx_enriched
    .withColumn('previous_datetime', F.lag('transaction_datetime').over(gap_window))
    .withColumn(
        'gap_minutes',
        (F.col('transaction_datetime').cast('long') - F.col('previous_datetime').cast('long')) / F.lit(60.0),
    )
)

pair_daily = (
    trx_enriched
    .groupBy('login_id', 'acct_nbr', 'calendar_date')
    .agg(
        F.count('*').alias('daily_touches'),
        F.sum((F.col('transaction_type') == '3').cast('int')).alias('daily_inquiries'),
        F.sum((F.col('transaction_type') == '4').cast('int')).alias('daily_maintenances'),
        F.sum('is_after_hours').alias('daily_after_hours_touches'),
        F.avg('gap_minutes').alias('daily_avg_gap_minutes'),
        (F.max('transaction_datetime').cast('long') - F.min('transaction_datetime').cast('long')).alias('daily_sequence_duration_seconds'),
    )
    .withColumn('daily_sequences', F.lit(1))
    .withColumn('daily_prop_inquiries', safe_div(F.col('daily_inquiries'), F.col('daily_touches')))
    .withColumn('daily_prop_maintenances', safe_div(F.col('daily_maintenances'), F.col('daily_touches')))
    .fillna(0, subset=['daily_avg_gap_minutes', 'daily_sequence_duration_seconds'])
)

## 3. Account calendar features

An extra 29 days is generated before the earliest sequence date so the first sequence row can still have a complete 30-day rolling balance standard deviation.

In [ ]:
account_calendar = (
    samples
    .groupBy('acct_nbr')
    .agg(
        F.date_sub(F.min(F.date_sub('anchor_date', WINDOW_DAYS - 1)), BALANCE_STD_DAYS - 1).alias('calendar_start'),
        F.max('anchor_date').alias('calendar_end'),
    )
    .withColumn('calendar_date', F.explode(F.sequence('calendar_start', 'calendar_end')))
    .select('acct_nbr', 'calendar_date')
)

acct_source = (
    df_acct
    .select(
        F.col('acct_nbr').cast('string').alias('acct_nbr'),
        F.to_date('effective_date').alias('effective_date'),
        F.coalesce(F.to_date('effective_end_date'), F.to_date(F.lit('9999-12-31'))).alias('effective_end_date'),
        F.col('acct_balance').cast('double').alias('daily_acct_balance'),
        F.lower(F.trim('acct_status_cd_description')).isin('active', 'normal').cast('int').alias('daily_is_normal'),
    )
    .dropna(subset=['acct_nbr', 'effective_date'])
)

account_match_window = Window.partitionBy(F.col('c.acct_nbr'), F.col('c.calendar_date')).orderBy(F.col('a.effective_date').desc())

account_daily = (
    account_calendar.alias('c')
    .join(
        acct_source.alias('a'),
        (F.col('c.acct_nbr') == F.col('a.acct_nbr'))
        & F.col('c.calendar_date').between(F.col('a.effective_date'), F.col('a.effective_end_date')),
        'left',
    )
    .withColumn('record_rank', F.row_number().over(account_match_window))
    .filter(F.col('record_rank') == 1)
    .select(
        F.col('c.acct_nbr').alias('acct_nbr'),
        F.col('c.calendar_date').alias('calendar_date'),
        'daily_acct_balance',
        'daily_is_normal',
    )
)

account_order = Window.partitionBy('acct_nbr').orderBy('calendar_date')
rolling_30d = account_order.rowsBetween(-(BALANCE_STD_DAYS - 1), 0)

account_daily = (
    account_daily
    .withColumn('previous_balance', F.lag('daily_acct_balance').over(account_order))
    .withColumn('previous_status', F.lag('daily_is_normal').over(account_order))
    .withColumn('daily_balance_change', F.coalesce(F.col('daily_acct_balance') - F.col('previous_balance'), F.lit(0.0)))
    .withColumn(
        'daily_status_change',
        F.when(F.col('previous_status').isNull(), F.lit(0))
        .otherwise((F.col('daily_is_normal') != F.col('previous_status')).cast('int')),
    )
    .withColumn('balance_std_30d', F.coalesce(F.stddev_pop('daily_acct_balance').over(rolling_30d), F.lit(0.0)))
    .drop('previous_balance', 'previous_status')
)

## 4. Employee and account peer features

In [ ]:
# Add each touched account's balance before aggregating across accounts.
pair_daily_with_balance = (
    pair_daily
    .join(
        account_daily.select('acct_nbr', 'calendar_date', 'daily_acct_balance'),
        ['acct_nbr', 'calendar_date'],
        'left',
    )
)

employee_daily = (
    pair_daily_with_balance
    .groupBy('login_id', 'calendar_date')
    .agg(
        F.sum('daily_touches').alias('employee_daily_total_touches'),
        F.sum('daily_sequences').alias('employee_daily_total_sequences'),
        F.countDistinct('acct_nbr').alias('employee_daily_accounts'),
        F.avg('daily_acct_balance').alias('employee_daily_mean_touched_balance'),
        F.stddev_pop('daily_acct_balance').alias('daily_stddev_acct_balance_emp'),
    )
    .withColumn(
        'employee_daily_avg_touches_per_account',
        safe_div(F.col('employee_daily_total_touches'), F.col('employee_daily_accounts')),
    )
    .fillna(0, subset=['daily_stddev_acct_balance_emp'])
)

account_peer_daily = (
    pair_daily
    .groupBy('acct_nbr', 'calendar_date')
    .agg(
        F.sum('daily_touches').alias('account_daily_total_touches'),
        F.sum('daily_sequences').alias('account_daily_total_sequences'),
    )
)

relative_daily = (
    pair_daily_with_balance
    .join(employee_daily, ['login_id', 'calendar_date'])
    .join(account_peer_daily, ['acct_nbr', 'calendar_date'])
    .withColumn(
        'daily_emp_prop_acct_seqs',
        safe_div(F.col('daily_sequences'), F.col('account_daily_total_sequences')),
    )
    # Employee's touch share on this account / employee's sequence share on this account.
    .withColumn(
        'daily_emp_touch_seq_ratio',
        safe_div(
            safe_div(F.col('daily_touches'), F.col('account_daily_total_touches')),
            safe_div(F.col('daily_sequences'), F.col('account_daily_total_sequences')),
        ),
    )
    # This account's touch share for the employee / its sequence share for the employee.
    .withColumn(
        'daily_acct_touch_seq_ratio',
        safe_div(
            safe_div(F.col('daily_touches'), F.col('employee_daily_total_touches')),
            safe_div(F.col('daily_sequences'), F.col('employee_daily_total_sequences')),
        ),
    )
    .withColumn(
        'daily_acct_balance_touch_deviation',
        F.col('daily_acct_balance') - F.col('employee_daily_mean_touched_balance'),
    )
    .withColumn(
        'daily_emp_behavior_deviation',
        F.col('daily_touches') - F.col('employee_daily_avg_touches_per_account'),
    )
)

## 5. Daily customer contact and address events

The helper records additions, removals and modifications after collapsing adjacent equal-value effective-date intervals. Multiple changes for the same party on the same day count as one party-day event, matching the existing project.

In [ ]:
def daily_change_events(df, id_cols, attr_cols):
    sentinel = F.to_date(F.lit('9999-12-31'))
    key_cols = id_cols + attr_cols
    work = df.select(
        *id_cols, *attr_cols,
        F.to_date('effective_date').alias('effective_date'),
        F.to_date('effective_end_date').alias('effective_end_date'),
    ).dropna(subset=id_cols + ['effective_date']).withColumn(
        '_end_filled', F.coalesce('effective_end_date', sentinel)
    )

    previous_rows = (
        Window.partitionBy(*key_cols)
        .orderBy('effective_date', '_end_filled')
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    grouped_rows = (
        Window.partitionBy(*key_cols)
        .orderBy('effective_date', '_end_filled')
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    work = (
        work
        .withColumn('_previous_max_end', F.max('_end_filled').over(previous_rows))
        .withColumn(
            '_new_group',
            F.when(F.col('_previous_max_end').isNull(), 1)
            .when(F.col('_previous_max_end') == sentinel, 0)
            .when(F.col('effective_date') > F.date_add('_previous_max_end', 1), 1)
            .otherwise(0),
        )
        .withColumn('_group', F.sum('_new_group').over(grouped_rows))
        .groupBy(*key_cols, '_group')
        .agg(
            F.min('effective_date').alias('effective_date'),
            F.max('_end_filled').alias('_end_filled'),
        )
        .withColumn(
            'effective_end_date',
            F.when(F.col('_end_filled') == sentinel, F.lit(None).cast('date'))
            .otherwise(F.col('_end_filled')),
        )
    )

    attr_value = F.col(attr_cols[0]) if len(attr_cols) == 1 else F.struct(*attr_cols)
    additions = work.select(*id_cols, F.col('effective_date').alias('calendar_date'), attr_value.alias('value'))
    removals = (
        work
        .filter(F.col('effective_end_date').isNotNull())
        .select(*id_cols, F.date_add('effective_end_date', 1).alias('calendar_date'), attr_value.alias('value'))
    )
    return additions.unionByName(removals).select(*id_cols, 'calendar_date').distinct()

address_fields = [
    'street_address', 'street_address_line_2', 'city', 'na_state_reg_prov_cd',
    'non_na_state_reg_prov', 'zip_code', 'country_cd'
]

clean_address = df_party_add
for c in address_fields:
    clean_address = clean_address.withColumn(c, F.lower(F.regexp_replace(F.coalesce(F.col(c), F.lit('')), r'\s+', '')))

clean_address = clean_address.withColumn('full_address', F.concat_ws('|', *address_fields))

primary_events = daily_change_events(
    clean_address.filter(F.lower('address_type') == 'primary address'),
    ['party_id'],
    ['full_address'],
)

mailing_events = daily_change_events(
    clean_address.filter(F.lower('address_type') == 'mailing address'),
    ['party_id', 'acct_id'],
    ['full_address'],
)

clean_contact = (
    df_party_contact
    .withColumn(
        'clean_contact_information',
        F.when(
            F.lower('contact_type') == 'phone',
            F.regexp_replace('contact_information', r'[^0-9]', ''),
        ).otherwise(F.lower(F.trim('contact_information'))),
    )
)

contact_events = daily_change_events(
    clean_contact,
    ['party_id'],
    ['contact_type', 'contact_subtype', 'clean_contact_information'],
)

acct_id_map = df_acct.select('acct_id', F.col('acct_nbr').cast('string').alias('acct_nbr')).dropna().distinct()
party_account = (
    df_p2a
    .join(acct_id_map, 'acct_id')
    .select(
        'party_id', 'acct_nbr',
        F.to_date('effective_date').alias('relationship_start'),
        F.coalesce(F.to_date('effective_end_date'), F.to_date(F.lit('9999-12-31'))).alias('relationship_end'),
    )
    .distinct()
)

def party_events_to_account(events):
    return (
        events.alias('e')
        .join(
            party_account.alias('p'),
            (F.col('e.party_id') == F.col('p.party_id'))
            & F.col('e.calendar_date').between(F.col('p.relationship_start'), F.col('p.relationship_end')),
            'inner',
        )
        .select(F.col('p.acct_nbr').alias('acct_nbr'), F.col('e.calendar_date').alias('calendar_date'), F.col('e.party_id').alias('party_id'))
        .distinct()
    )

contact_account_daily = (
    party_events_to_account(contact_events)
    .groupBy('acct_nbr', 'calendar_date')
    .agg(F.count('*').alias('daily_contact_info_changes'))
)

primary_account_daily = (
    party_events_to_account(primary_events)
    .groupBy('acct_nbr', 'calendar_date')
    .agg(F.count('*').alias('daily_primary_address_changes'))
)

mailing_account_daily = (
    mailing_events
    .join(acct_id_map, 'acct_id')
    .select('acct_nbr', 'calendar_date', 'party_id')
    .distinct()
    .groupBy('acct_nbr', 'calendar_date')
    .agg(F.count('*').alias('daily_mailing_address_changes'))
)

address_account_daily = (
    primary_account_daily
    .join(mailing_account_daily, ['acct_nbr', 'calendar_date'], 'full')
    .fillna(0, subset=['daily_primary_address_changes', 'daily_mailing_address_changes'])
    .withColumn(
        'daily_address_changes',
        F.col('daily_primary_address_changes') + F.col('daily_mailing_address_changes'),
    )
    .select('acct_nbr', 'calendar_date', 'daily_address_changes')
)

## 6. Assemble the 20 features

In [ ]:
behavior_fill_cols = [
    'daily_touches', 'daily_sequences', 'daily_inquiries', 'daily_maintenances',
    'daily_prop_inquiries', 'daily_prop_maintenances', 'daily_after_hours_touches',
    'daily_avg_gap_minutes', 'daily_sequence_duration_seconds',
    'employee_daily_avg_touches_per_account', 'daily_stddev_acct_balance_emp',
    'daily_acct_balance_touch_deviation', 'daily_emp_behavior_deviation',
    'daily_emp_prop_acct_seqs', 'daily_emp_touch_seq_ratio', 'daily_acct_touch_seq_ratio',
    'daily_contact_info_changes', 'daily_address_changes',
]

daily_features_long = (
    spine
    .join(relative_daily.drop('daily_acct_balance'), ['login_id', 'acct_nbr', 'calendar_date'], 'left')
    .join(
        account_daily.select(
            'acct_nbr', 'calendar_date', 'daily_acct_balance',
            'daily_balance_change', 'daily_status_change', 'balance_std_30d',
        ),
        ['acct_nbr', 'calendar_date'],
        'left',
    )
    .join(contact_account_daily, ['acct_nbr', 'calendar_date'], 'left')
    .join(address_account_daily, ['acct_nbr', 'calendar_date'], 'left')
    .fillna(0, subset=behavior_fill_cols)
    .select(*(sample_key + metadata_cols + ['calendar_date', 'sequence_day'] + FEATURE_COLS))
)

daily_features_long.orderBy(*sample_key, 'sequence_day').display()

## 7. Validation and LSTM-ready nested arrays

Validation fails early if a sample does not contain exactly 90 unique days, if a daily key is duplicated, or if account history is missing.

In [ ]:
duplicate_count = (
    daily_features_long
    .groupBy(*(sample_key + ['calendar_date']))
    .count()
    .filter(F.col('count') != 1)
    .count()
)
assert duplicate_count == 0, f'Found {duplicate_count} duplicate daily sample keys'

bad_length_count = (
    daily_features_long
    .groupBy(*sample_key)
    .agg(F.count('*').alias('rows'), F.countDistinct('calendar_date').alias('days'))
    .filter((F.col('rows') != WINDOW_DAYS) | (F.col('days') != WINDOW_DAYS))
    .count()
)
assert bad_length_count == 0, f'Found {bad_length_count} samples without exactly {WINDOW_DAYS} days'

missing_account_count = daily_features_long.filter(F.col('daily_acct_balance').isNull()).count()
assert missing_account_count == 0, f'Found {missing_account_count} rows without an effective account balance'

null_feature_exprs = [
    F.sum(F.col(c).isNull().cast('int')).alias(c) for c in FEATURE_COLS
]
null_counts = daily_features_long.agg(*null_feature_exprs).first().asDict()
assert sum(null_counts.values()) == 0, f'Null feature values found: {null_counts}'

different_ratio_rows = daily_features_long.filter(
    F.abs(F.col('daily_emp_touch_seq_ratio') - F.col('daily_acct_touch_seq_ratio')) > 1e-12
).count()
print('Rows where the two ratio features differ:', different_ratio_rows)

feature_vector = F.array(*[F.col(c).cast('double') for c in FEATURE_COLS])
metadata_for_sequence = [c for c in metadata_cols if c not in ['job_family_description', 'party_ids']]

lstm_sequences = (
    daily_features_long
    .withColumn('day_data', F.struct('sequence_day', 'calendar_date', feature_vector.alias('features')))
    .groupBy(*(sample_key + metadata_for_sequence))
    .agg(F.sort_array(F.collect_list('day_data')).alias('days'))
    .withColumn('calendar_dates', F.transform('days', lambda x: x['calendar_date']))
    .withColumn('daily_feature_matrix', F.transform('days', lambda x: x['features']))
    .withColumn('feature_names', F.array(*[F.lit(c) for c in FEATURE_COLS]))
    .drop('days')
)

lstm_sequences.select(*sample_key, F.size('daily_feature_matrix').alias('sequence_length'), 'daily_feature_matrix').display()
print('Feature order:', FEATURE_COLS)

## 8. Optional output

Set `WRITE_OUTPUT = True` in the configuration cell to overwrite the two versioned Delta outputs.

In [ ]:
if WRITE_OUTPUT:
    (
        daily_features_long
        .write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .save(long_output_path)
    )
    (
        lstm_sequences
        .write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .save(sequence_output_path)
    )
    print('Saved:', long_output_path)
    print('Saved:', sequence_output_path)
else:
    print('WRITE_OUTPUT=False: dataframes were generated and validated but not saved.')